In [1]:
from com.example.agentic.agent.LLMManager import LLMManager

llm = LLMManager.create_llm()
llm.call("Hello")

'Hello! How can I help you today?'

In [2]:
import os
import json
from crewai.tools import tool

@tool("Image url extract tool")
def image_fetch_tool(doc_path: str) -> str:
    """Tool to extract image url from json file."""
    tool_output = "Design Reference Images:\n\n"
    try:
        work_dir = os.getenv("WORK_DIR")
        with open(doc_path, 'r') as file:
            _json = json.load(file)
            
        _images = _json.get('design_reference_images', [])
        formatted = "Design Reference Images:\n\n"
        for _image in _images:
            tool_output += f"""
                title: {_image['title']}
                url: {_image['url']}
                -------------------"""
    except Exception as e:
        raise ValueError(f"Failed to fetch space news: {str(e)}")
    #
    return tool_output

#result = image_fetch_tool('docs/json/designs-research.json')
#print(result)

In [3]:
from pydantic import BaseModel, Field
from typing import List, Dict


class Image(BaseModel):
    title: str = Field(description="The title of the image")
    url: str = Field(description="The url of the image")

class ImagesOutput(BaseModel):
    topic: str = Field(description="The details about primary topic")    
    images: List[Image] = Field(description="List of top images on topic")

In [4]:
from crewai import LLM, Agent, Task, Crew, Process
from crewai_tools import FileReadTool, JSONSearchTool

# 2. Agent to process the file
file_reader = Agent(
    role='Data Extraction Specialist',
    goal='Extract structured records from the provided image_fetch_tool',
    backstory="You are an expert at parsing unstructured text into clean data.",
    tools=[image_fetch_tool],
    max_iter=3,
    verbose=True,
    allow_delegation=False,
)

# 4. Create the Task
extraction_task = Task(
    description='Read the {image_file} file and extract all records into a list.',
    expected_output="A structured object that should contain following "
    "- topic : represent provided {topic} "
    "- images : list of records containing title and url."
    "make sure output contain minimume 5 records in images and validate that results records should match values from file {image_file}",
    agent=file_reader,
    tools=[image_fetch_tool],
    output_pydantic=ImagesOutput,
    output_file="outputs/L006/extraction_task_{run_id}.json" 
)


# 4. Run the Crew
crew = Crew(
    agents=[file_reader],
    tasks=[extraction_task],
    process=Process.sequential,
    verbose=True,
    #llm=llm
)

from datetime import datetime

work_dir = os.getenv("WORK_DIR")
run_date = datetime.now().strftime('%Y-%m-%d')
run_id   = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Crew triggered on {run_date} with execution id {run_id}")

inputs = {
    "topic":"Microservice Design",
    "image_file": f"{work_dir}/docs/json/designs-research.json",
    "run_date": run_date,
    "run_id": run_id,
}

# Execute the Tasks
result = crew.kickoff(inputs=inputs)
print(result)

Crew triggered on 2026-04-30 with execution id 20260430_083107


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 17b7d4c3-6d9b-477a-9d50-97a2651f217d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Read the /home/brijeshdhaker/IdeaProjects/crewai_design_document/docs/json/designs-research.json file    │
│  and extract all records into a list.                                                                           │
│  ID: 54f637b3-c497-4294-99a9-3859c7d4d057                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Extraction Specialist                                                                              │
│                                                                                                                 │
│  Task: Read the /home/brijeshdhaker/IdeaProjects/crewai_design_document/docs/json/designs-research.json file    │
│  and extract all records into a list.                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool image_url_extract_tool executed with result: Design Reference Images:


                title: Design Microservices Architecture with Patterns & Principles | Medium
                url: https://miro.medium.com/1*JIDAhbsGGTztmcJ6OxNkrg.png
      ...


...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: image_url_extract_tool                                                                                   │
│  Output: Design Reference Images:                                                                               │
│                                                                                                                 │
│                                                                                                                 │
│                  title: Design Microservices Architecture with Patterns & Principles | Medium                   │
│                  url: https://miro.medium.com/1*JIDAhbsGGTztmcJ6OxNkrg.png                                      │
│                  -------------------                                                                            │
│                  title: Microsoft Microservices Architecture Style                                              │
│                  url: https://miro.medium.com/v2/resize:fit:1400/1*6pdGag7cDTYOhJ6h9D2HHg.png                   │
│                  -------------------                                                                            │
│                  title: A Pragmatic Journey from Monolith to Microservices                                      │
│                  url: https://miro.medium.com/v2/resize:fit:4800/format:webp/1*P8bRYocXcVi_82R49eOckA.png       │
│                  -------------------                                                                            │
│                  title: Microservices Architecture for eCommerce Domain                                         │
│                  url:                                                                                           │
│  https://www.simform.com/wp-content/uploads/2023/12/Microservices-Architecture-for-an-eCommerce-App.png         │
│                  -------------------                                                                            │
│                  title: Microservices Design principle and Design Patterns                                      │
│                  url: https://miro.medium.com/v2/resize:fit:1400/0*CVxZxfbawqA_Ln2G.png                         │
│                  -------------------                                                                            │
│                  title: Microservices Architecture: Simplified Diagram and Benefits                             │
│                  url:                                                                                           │
│  https://media.licdn.com/dms/image/v2/D4D22AQEw6HJaKuC0HA/feedshare-shrink_1280/B4DZvsnx8lI0Ac-/0/176920141179  │
│  3?e=2147483647&v=beta&t=KzF_9W1S6k-DC-jvnMqy03mwZUThOKKTccGE0YlN54o                                            │
│                  -------------------                                                                            │
│                  title: Typical Microservice Architecture                                                       │
│                  url: https://assets.bytebytego.com/diagrams/0396-typical-microservice-architecture.png         │
│                  -------------------                                                                            │
│                  title: Oracle Microservices Deployment                                                         │
│                  url:                                                                                           │
│  https://docs.oracle.com/en/solutions/learn-architect-microservice/img/microservice_architecture.png            │
│                  -------------------                   

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Extraction Specialist                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  topic='Design Microservices Architecture with Patterns & Principles' images=[Image(title='Design               │
│  Microservices Architecture with Patterns & Principles | Medium',                                               │
│  url='https://miro.medium.com/1*JIDAhbsGGTztmcJ6OxNkrg.png'), Image(title='Microsoft Microservices              │
│  Architecture Style', url='https://miro.medium.com/v2/resize:fit:1400/1*6pdGag7cDTYOhJ6h9D2HHg.png'),           │
│  Image(title='A Pragmatic Journey from Monolith to Microservices',                                              │
│  url='https://miro.medium.com/v2/resize:fit:4800/format:webp/1*P8bRYocXcVi_82R49eOckA.png'),                    │
│  Image(title='Microservices Architecture for eCommerce Domain',                                                 │
│  url='https://www.simform.com/wp-content/uploads/2023/12/Microservices-Architecture-for-an-eCommerce-App.png')  │
│  , Image(title='Microservices Design principle and Design Patterns',                                            │
│  url='https://miro.medium.com/v2/resize:fit:1400/0*CVxZxfbawqA_Ln2G.png'), Image(title='Microservices           │
│  Architecture: Simplified Diagram and Benefits',                                                                │
│  url='https://media.licdn.com/dms/image/v2/D4D22AQEw6HJaKuC0HA/feedshare-shrink_1280/B4DZvsnx8lI0Ac-/0/1769201  │
│  411793?e=2147483647&v=beta&t=KzF_9W1S6k-DC-jvnMqy03mwZUThOKKTccGE0YlN54o'), Image(title='Typical Microservice  │
│  Architecture', url='https://assets.bytebytego.com/diagrams/0396-typical-microservice-architecture.png'),       │
│  Image(title='Oracle Microservices Deployment',                                                                 │
│  url='https://docs.oracle.com/en/solutions/learn-architect-microservice/img/microservice_architecture.png'),    │
│  Image(title='Microservices Architecture Diagram Guide : Concepts, Creation Tutorials, and Templates',          │
│  url='https://cdn.processon.io/admin/knowledge/article_content_img/67b6a344bcb4581cedd12229.png'),              │
│  Image(title='Microservices API Gateway Desing',                                                                │
│  url='https://assets.testmuai.com/resources/uploads/2024/03/Architecture-1.png')]                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Read the /home/brijeshdhaker/IdeaProjects/crewai_design_document/docs/json/designs-research.json file    │
│  and extract all records into a list.                                                                           │
│  Agent: Data Extraction Specialist                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 17b7d4c3-6d9b-477a-9d50-97a2651f217d                                                                       │
│  Final Output: {"topic":"Design Microservices Architecture with Patterns &                                      │
│  Principles","images":[{"title":"Design Microservices Architecture with Patterns & Principles |                 │
│  Medium","url":"https://miro.medium.com/1*JIDAhbsGGTztmcJ6OxNkrg.png"},{"title":"Microsoft Microservices        │
│  Architecture                                                                                                   │
│  Style","url":"https://miro.medium.com/v2/resize:fit:1400/1*6pdGag7cDTYOhJ6h9D2HHg.png"},{"title":"A Pragmatic  │
│  Journey from Monolith to                                                                                       │
│  Microservices","url":"https://miro.medium.com/v2/resize:fit:4800/format:webp/1*P8bRYocXcVi_82R49eOckA.png"},{  │
│  "title":"Microservices Architecture for eCommerce                                                              │
│  Domain","url":"https://www.simform.com/wp-content/uploads/2023/12/Microservices-Architecture-for-an-eCommerce  │
│  -App.png"},{"title":"Microservices Design principle and Design                                                 │
│  Patterns","url":"https://miro.medium.com/v2/resize:fit:1400/0*CVxZxfbawqA_Ln2G.png"},{"title":"Microservices   │
│  Architecture: Simplified Diagram and                                                                           │
│  Benefits","url":"https://media.licdn.com/dms/image/v2/D4D22AQEw6HJaKuC0HA/feedshare-shrink_1280/B4DZvsnx8lI0A  │
│  c-/0/1769201411793?e=2147483647&v=beta&t=KzF_9W1S6k-DC-jvnMqy03mwZUThOKKTccGE0YlN54o"},{"title":"Typical       │
│  Microservice                                                                                                   │
│  Architecture","url":"https://assets.bytebytego.com/diagrams/0396-typical-microservice-architecture.png"},{"ti  │
│  tle":"Oracle Microservices                                                                                     │
│  Deployment","url":"https://docs.oracle.com/en/solutions/learn-architect-microservice/img/microservice_archite  │
│  cture.png"},{"title":"Microservices Architecture Diagram Guide : Concepts, Creation Tutorials, and             │
│  Templates","url":"https://cdn.processon.io/admin/knowledge/article_content_img/67b6a344bcb4581cedd12229.png"}  │
│  ,{"title":"Microservices API Gateway                                                                           │
│  Desing","url":"https://assets.testmuai.com/resources/uploads/2024/03/Architecture-1.png"}]}                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

topic='Design Microservices Architecture with Patterns & Principles' images=[Image(title='Design Microservices Architecture with Patterns & Principles | Medium', url='https://miro.medium.com/1*JIDAhbsGGTztmcJ6OxNkrg.png'), Image(title='Microsoft Microservices Architecture Style', url='https://miro.medium.com/v2/resize:fit:1400/1*6pdGag7cDTYOhJ6h9D2HHg.png'), Image(title='A Pragmatic Journey from Monolith to Microservices', url='https://miro.medium.com/v2/resize:fit:4800/format:webp/1*P8bRYocXcVi_82R49eOckA.png'), Image(title='Microservices Architecture for eCommerce Domain', url='https://www.simform.com/wp-content/uploads/2023/12/Microservices-Architecture-for-an-eCommerce-App.png'), Image(title='Microservices Design principle and Design Patterns', url='https://miro.medium.com/v2/resize:fit:1400/0*CVxZxfbawqA_Ln2G.png'), Image(title='Microservices Architecture: Simplified Diagram and Benefits', url='https://media.licdn.com/dms/image/v2/D4D22AQEw6HJaKuC0HA/feedshare-shrink_1280/B4DZvsnx

##### GitHub Tool

In [ ]:
from github import Github, Auth
from langchain_community.agent_toolkits.github.toolkit import GitHubToolkit
from langchain_community.utilities.github import GitHubAPIWrapper

github = GitHubAPIWrapper()
toolkit = GitHubToolkit.from_github_api_wrapper(github)

_repo = 'brijeshdhaker/docker-hadoop-cluster'
_git_hub_token = os.getenv("GIT_HUB_TOKEN")

# Use the new Auth class to handle your token
auth = Auth.Token(_git_hub_token)

# Authenticate using a Personal Access Token
github = Github(auth=auth)

# Get a specific repository
repo = github.get_repo(_repo)

print(repo.stargazers_count)

In [ ]:
from langchain_community.agent_toolkits.github.toolkit import GitHubToolkit
from langchain_community.utilities.github import GitHubAPIWrapper

_repo = 'brijeshdhaker/docker-hadoop-cluster'
_git_hub_token = os.getenv("GIT_HUB_TOKEN")

github = GitHubAPIWrapper(
    active_branch="main", 
    github_repository="brijeshdhaker/docker-hadoop-cluster",
    
)
# The wrapper automatically looks for GITHUB_PERSONAL_ACCESS_TOKEN if not passed
toolkit = GitHubToolkit.from_github_api_wrapper(github)
tools = toolkit.get_tools()
for tool in tools:
    print(tool.name)

In [ ]:
from langchain_community.document_loaders import GithubFileLoader

_repo = 'brijeshdhaker/docker-hadoop-cluster'
_git_hub_token = os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")

loader = GithubFileLoader(
    repo=_repo,  # the repo name
    branch="main",  # the branch name
    access_token=_git_hub_token,
    github_api_url="https://api.github.com",
    file_filter=lambda file_path: file_path.endswith(
        ".md"
    ),  # load all markdowns files.
)
documents = loader.load()

In [ ]:
documents[1].page_content

In [ ]:
from crewai_tools import GithubSearchTool

_repo = 'brijeshdhaker/docker-hadoop-cluster'
_git_hub_token = os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")

# Initialize the tool with your PAT
github_tool = GithubSearchTool(
    config=_rag_tool_config,
    github_repo='https://github.com/brijeshdhaker/docker-hadoop-cluster',
    gh_token=_git_hub_token,
    content_types=['code', 'issue']
)

# Use the tool within an Agent
# from crewai import Agent

# github_specialist = Agent(
#     role='GitHub Researcher',
#     goal='Search for specific code patterns and issues in the repository',
#     backstory='Expert in navigating complex codebases and tracking development issues.',
#     tools=[github_tool],
#     verbose=True
# )




In [ ]:
github_tool.add(repo='brijeshdhaker/docker-hadoop-cluster',content_types=['code', 'issue'])
github_tool.run('Search for specific code patterns and issues in the repository')

In [ ]:
# Import necessary modules
from crewai import Agent, Task, Crew
#from ChatOpenAI import ChatOpenAI
import os

from crewai.tools import tool

@tool("calculator")
def calculator(query: str) -> str:
    """A simple calculator tool that evaluates math expressions."""
    try:
        result = eval(query)
        return f"The result of {query} is {result}"
    except Exception as e:
        return f"Error: {str(e)}"

from crewai import Agent

researcher = Agent(
    role="Researcher",
    goal="Perform research and compute results using tools.",
    backstory="Handles calculations and prepares data for reporting.",
    tools=[calculator],
    verbose=True
)

writer = Agent(
    role="Writer",
    goal="Summarize the research into a readable article.",
    backstory="Takes calculation results and writes summaries for users.",
    verbose=True
)

from crewai import Task

task1 = Task(
    description="Calculate '12 * (5 + 3)' using the calculator tool.",
    agent=researcher,
    expected_output="The numeric result of the expression."
)

task2 = Task(
    description="Summarize the researcher's calculation in a short article.",
    agent=writer,
    expected_output="A summary article including the calculation result.",
    context=[task1]  # Provides the output of task1 as context
)

# Initialize the language model from OpenAI with the specified base URL and model
model = LLMManager.create_llm('ollama')

# Define the agent's role, goal, backstory, and settings
general_agent = Agent(
    role="Math Professor",
    goal="""Provide solutions to students asking mathematical questions, 
            explaining answers clearly and understandably.""",
    backstory="""You are a skilled math professor who enjoys explaining math 
                 problems in a way that students can easily follow.""",
    allow_delegation=False,
    verbose=False,
    llm=model
)

# Define a task for the agent to solve
task = Task(
    description="What is 3 + 5?",
    agent=general_agent,
    expected_output="A numerical answer."
)

# Initialize the crew with the agent and task, setting verbosity
# crew = Crew(
#     agents=[general_agent],
#     tasks=[task],
#     verbose=True
# )

from crewai import Crew, Process

crew = Crew(
    agents=[researcher, writer],
    tasks=[task1, task2],
    process=Process.sequential,  # Executes tasks in order
    verbose=True
)

result = crew.kickoff()
print("\nFinal Result:\n", result)



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 621ea6ec-e1c5-4c8b-89bb-1c1beb6174f5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Calculate '12 * (5 + 3)' using the calculator tool.                                                      │
│  ID: c8a3c1db-337d-4eec-9b0a-9ce09a9d3e42                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Researcher                                                                                              │
│                                                                                                                 │
│  Task: Calculate '12 * (5 + 3)' using the calculator tool.                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool calculator executed with result: The result of 12 * (5 + 3) is 96...


╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: calculator                                                                                               │
│  Args: {'query': '12 * (5 + 3)'}                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Writer                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Research Calculation Summary**                                                                               │
│                                                                                                                 │
│  This document summarizes the researcher's final calculation results based on the provided data points. The     │
│  analysis concludes the following key finding regarding the calculation performed.                              │
│                                                                                                                 │
│  ### Key Findings                                                                                               │
│                                                                                                                 │
│  The calculated result for this study is **96**. This integer represents the definitive outcome of the          │
│  mathematical procedures applied during the assessment. The researcher utilized the established methods to      │
│  derive this final value, ensuring accuracy within the data structure provided.                                 │
│                                                                                                                 │
│  ### Conclusion                                                                                                 │
│                                                                                                                 │
│  The study concludes with a calculated value of 96. This result is the primary metric identified, indicating    │
│  the final sum or score derived from the dataset analysis. The findings validate that 96 is the correct final   │
│  output for this calculation task.                                                                              │
│                                                                                                                 │
│  **Final Result: 96**                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Summarize the researcher's calculation in a short article.                                               │
│  Agent: Writer                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Final Result:
 **Research Calculation Summary**

This document summarizes the researcher's final calculation results based on the provided data points. The analysis concludes the following key finding regarding the calculation performed.

### Key Findings

The calculated result for this study is **96**. This integer represents the definitive outcome of the mathematical procedures applied during the assessment. The researcher utilized the established methods to derive this final value, ensuring accuracy within the data structure provided.

### Conclusion

The study concludes with a calculated value of 96. This result is the primary metric identified, indicating the final sum or score derived from the dataset analysis. The findings validate that 96 is the correct final output for this calculation task.

**Final Result: 96**


╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ 6e00edb2-d403-4d5f-b836-1c4cb1c82f62                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/6e00edb2-d403-4d5 │
│ f-b836-1c4cb1c82f62?access_code=TRACE-f978c25868                             │
│ 🔑 Access Code: TRACE-f978c25868                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 621ea6ec-e1c5-4c8b-89bb-1c1beb6174f5                                                                       │
│  Final Output: **Research Calculation Summary**                                                                 │
│                                                                                                                 │
│  This document summarizes the researcher's final calculation results based on the provided data points. The     │
│  analysis concludes the following key finding regarding the calculation performed.                              │
│                                                                                                                 │
│  ### Key Findings                                                                                               │
│                                                                                                                 │
│  The calculated result for this study is **96**. This integer represents the definitive outcome of the          │
│  mathematical procedures applied during the assessment. The researcher utilized the established methods to      │
│  derive this final value, ensuring accuracy within the data structure provided.                                 │
│                                                                                                                 │
│  ### Conclusion                                                                                                 │
│                                                                                                                 │
│  The study concludes with a calculated value of 96. This result is the primary metric identified, indicating    │
│  the final sum or score derived from the dataset analysis. The findings validate that 96 is the correct final   │
│  output for this calculation task.                                                                              │
│                                                                                                                 │
│  **Final Result: 96**                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [7]:
import random
import ollama

def get_lucky_number(min_number: int, max_number: int) -> int:
    """
    Generate Lucky number to use in your calculations

    Args:
         min_number: Minimum integer value
         max_number: Maximum integer value

    Returns:
        int: Randomly generated number between minimum and maximum limit
    """
    lucky_number = random.randint(int(min_number), int(max_number))
    return lucky_number


def this_is_dummy_function_to_confuse_llm(min_number: int, max_number: int) -> int:
    """
    This is a dummy function to confuse llm

    Args:
         min_number: Minimum integer value
         max_number: Maximum integer value

    Returns:
        int: a static integer value
    """
    return 5

available_functions = {
    'get_lucky_number': get_lucky_number,
    'this_is_dummy_function_to_confuse_llm': this_is_dummy_function_to_confuse_llm,
}




In [ ]:

response = ollama.chat(
    'llama3.2',
    messages=[{
        'role': 'user',
        'content': 'Generate a lucky number between 13 and 20'
    }],
    tools=[get_lucky_number, this_is_dummy_function_to_confuse_llm]
)
for tool in response.message.tool_calls or []:
    function_to_call = available_functions.get(tool.function.name)
    if function_to_call:
        print('Function output:', function_to_call(**tool.function.arguments))
    else:
        print('Function not found:', tool.function.name)

Function output: 16
